In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
class PINN(nn.Module):
    def __init__(self, width=32, depth=3):
        super().__init__()

        layers = []
        layers.append(nn.Linear(2, width))
        layers.append(nn.Tanh())

        for _ in range(depth-1):
            layers.append(nn.Linear(width, width))
            layers.append(nn.Tanh())

        layers.append(nn.Linear(width, 1))

        self.net = nn.Sequential(*layers)

    def forward(self, xy):
        return self.net(xy)

In [ ]:
def sample_interior(N):
    return torch.rand(N, 2, device=device)


def sample_boundary(N):
    N4 = N // 4
    t = torch.rand(N4, 1, device=device)

    bottom = torch.cat([t, torch.zeros_like(t)], dim=1)
    top    = torch.cat([t, torch.ones_like(t)], dim=1)
    left   = torch.cat([torch.zeros_like(t), t], dim=1)
    right  = torch.cat([torch.ones_like(t), t], dim=1)

    return torch.cat([bottom, top, left, right], dim=0)

In [ ]:
def laplacian(model, xy):
    xy.requires_grad_(True)

    u = model(xy)

    grad_u = torch.autograd.grad(
        u, xy,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    u_x = grad_u[:, 0:1]
    u_y = grad_u[:, 1:2]

    u_xx = torch.autograd.grad(
        u_x, xy,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True
    )[0][:, 0:1]

    u_yy = torch.autograd.grad(
        u_y, xy,
        grad_outputs=torch.ones_like(u_y),
        create_graph=True
    )[0][:, 1:2]

    return u_xx + u_yy

In [ ]:
def loss_pinn(model, xy_r, xy_b, lambda_bc=100.0):
    delta_u = laplacian(model, xy_r)

    # PDE: -Delta u = 1
    residual = -delta_u - 1.0

    loss_pde = torch.mean(residual**2)

    # Boundary condition: u = 0
    u_b = model(xy_b)
    loss_bc = torch.mean(u_b**2)

    return loss_pde + lambda_bc*loss_bc, loss_pde, loss_bc

In [ ]:
model = PINN(width=32, depth=3).to(device)

xy_r = sample_interior(3000)
xy_b = sample_boundary(800)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5001):
    optimizer.zero_grad()

    loss, lpde, lbc = loss_pinn(model, xy_r, xy_b)

    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(
            f"epoch {epoch:5d} | "
            f"loss = {loss.item():.3e} | "
            f"PDE = {lpde.item():.3e} | "
            f"BC = {lbc.item():.3e}"
        )

In [ ]:
n = 100

xs = np.linspace(0, 1, n)
ys = np.linspace(0, 1, n)

X, Y = np.meshgrid(xs, ys, indexing="ij")

xy_np = np.stack([X.reshape(-1), Y.reshape(-1)], axis=1)
xy = torch.tensor(xy_np, dtype=torch.float64, device=device)

with torch.no_grad():
    U = model(xy).cpu().numpy().reshape(n, n)

plt.figure(figsize=(6,5))
plt.contourf(X, Y, U, levels=40)
plt.colorbar()
plt.title("PINN solution of Poisson problem")
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.show()

In [ ]:
from ngsolve import *
from netgen.geom2d import unit_square
from ngsolve.webgui import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.01))

fes = H1(mesh, order=4, dirichlet=".*")

u, v = fes.TnT()

a = BilinearForm(grad(u)*grad(v)*dx)
f = LinearForm(1*v*dx)

gfu = GridFunction(fes, name="FEM solution")

with TaskManager():
    a.Assemble()
    f.Assemble()
    gfu.vec.data = a.mat.Inverse(fes.FreeDofs()) * f.vec

Draw(gfu, mesh, "FEM solution");

In [ ]:
n = 100

xs = np.linspace(0, 1, n)
ys = np.linspace(0, 1, n)

X, Y = np.meshgrid(xs, ys, indexing="ij")
points = np.stack([X.reshape(-1), Y.reshape(-1)], axis=1)

U_fem = np.array([
    gfu(mesh(x, y))
    for x, y in points
]).reshape(n, n)

xy_torch = torch.tensor(points, dtype=torch.float64, device=device)

with torch.no_grad():
    U_pinn = model(xy_torch).cpu().numpy().reshape(n, n)

Diff = U_pinn - U_fem

l2_diff = np.sqrt(np.mean(Diff**2))
max_diff = np.max(np.abs(Diff))
rel_l2_diff = l2_diff / np.sqrt(np.mean(U_fem**2))

print("Grid L2 difference          =", l2_diff)
print("Relative grid L2 difference =", rel_l2_diff)
print("Max difference              =", max_diff)

plt.figure(figsize=(6,5))
plt.contourf(X, Y, Diff, levels=40)
plt.colorbar()
plt.title("PINN - FEM")
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.show()

plt.figure(figsize=(6,5))
plt.contourf(X, Y, np.abs(Diff), levels=40)
plt.colorbar()
plt.title("|PINN - FEM|")
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.show()